# Preprocessing and model checks

Exploratory preprocessing and baseline model checks for the pyrite metallogenic typing workflow.

**Input data:** place the standardized Excel file at `../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx`.

**Note:** outputs were cleared for GitHub readability and reproducibility.


In [ ]:
import pandas as pd
import numpy as np

# Read your Excel file into a Pandas DataFrame
df = pd.read_excel('../data/raw/Pyrite_23_April_Final_Version_New_Paper.xlsx')
# Specify columns to transformPyrite_23_April (Final Version) New Paper.xlsx
columns_to_transform = ['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']

# Apply natural logarithm to selected columns
df[columns_to_transform] = np.log(df[columns_to_transform])

# Calculate the mean and standard deviation of the log-transformed columns
mean = df[columns_to_transform].mean()
std = df[columns_to_transform].std()

# Standardize the log-transformed columns using the mean and standard deviation
df[columns_to_transform] = (df[columns_to_transform] - mean) / std



# Save the result to a new Excel file
df.to_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx', index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Load data
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Define features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X_imputed, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Function to evaluate and report metrics including AUC
def train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, technique_name):
    print(f'\n==== {technique_name} ====')
    
    # Train model
    rf = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)
    rf.fit(X_train, y_train)
    
    # Validation metrics
    y_val_pred = rf.predict(X_val)
    y_val_proba = rf.predict_proba(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')
    
    print(f'\nValidation Accuracy: {val_accuracy * 100:.2f}%')
    print(f'Validation AUC (OvR): {val_auc:.4f}')
    print('Validation Confusion Matrix:\n', confusion_matrix(y_val, y_val_pred))
    print('Validation Classification Report:\n', classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))
    
    # Test metrics
    y_test_pred = rf.predict(X_test)
    y_test_proba = rf.predict_proba(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f'\nTest Accuracy: {test_accuracy * 100:.2f}%')
    print(f'Test AUC (OvR): {test_auc:.4f}')
    print('Test Confusion Matrix:\n', confusion_matrix(y_test, y_test_pred))
    print('Test Classification Report:\n', classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# === 1. Original (No Resampling)
train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, "Original (No Resampling)")

# === 2. SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
train_and_evaluate(X_train_smote, y_train_smote, X_val, y_val, X_test, y_test, "SMOTE (Over-sampling)")

# === 3. RUS
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
train_and_evaluate(X_train_rus, y_train_rus, X_val, y_val, X_test, y_test, "Random Under-Sampling (RUS)")


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set seed
seed = 42
random.seed(seed)
np.random.seed(seed)

# Load dataset
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Define features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Function to evaluate a model
def evaluate_svm_model(X_data, y_data, label, C=100, gamma=0.1):
    # Train-validation-test split
    X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.4, random_state=seed, stratify=y_data)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

    # Train SVM with probability=True
    svm_classifier = SVC(C=C, kernel='rbf', gamma=gamma, probability=True, random_state=seed)
    svm_classifier.fit(X_train, y_train)

    # Validation
    y_val_pred = svm_classifier.predict(X_val)
    y_val_proba = svm_classifier.predict_proba(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')

    print(f"\n--- {label} | Validation ---")
    print(f"Accuracy: {accuracy_score(y_val, y_val_pred) * 100:.2f}%")
    print(f"AUC (OvR): {val_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
    print("Classification Report:\n", classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

    # Test
    y_test_pred = svm_classifier.predict(X_test)
    y_test_proba = svm_classifier.predict_proba(X_test)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f"\n--- {label} | Test ---")
    print(f"Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")
    print(f"AUC (OvR): {test_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report:\n", classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# 1. Original
evaluate_svm_model(X_imputed, y_encoded, "Original (No Resampling)")

# 2. SMOTE
smote = SMOTE(random_state=seed)
X_smote, y_smote = smote.fit_resample(X_imputed, y_encoded)
evaluate_svm_model(X_smote, y_smote, "SMOTE Resampling")

# 3. RUS
rus = RandomUnderSampler(random_state=seed)
X_rus, y_rus = rus.fit_resample(X_imputed, y_encoded)
evaluate_svm_model(X_rus, y_rus, "Random UnderSampling (RUS)")


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set random seed
seed = 42
random.seed(seed)
np.random.seed(seed)

# Load dataset
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Function to train & evaluate GB model with AUC
def evaluate_gb_model(X_data, y_data, label):
    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.4, random_state=seed, stratify=y_data)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

    # Initialize and train model
    gb_classifier = GradientBoostingClassifier(
        learning_rate=0.1,
        max_depth=4,
        n_estimators=300,
        random_state=seed
    )
    gb_classifier.fit(X_train, y_train)

    # Validation predictions
    y_val_pred = gb_classifier.predict(X_val)
    y_val_proba = gb_classifier.predict_proba(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')

    print(f"\n--- {label} | Validation ---")
    print(f"Accuracy: {accuracy_score(y_val, y_val_pred) * 100:.2f}%")
    print(f"AUC (OvR): {val_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
    print("Classification Report:\n", classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

    # Test predictions
    y_test_pred = gb_classifier.predict(X_test)
    y_test_proba = gb_classifier.predict_proba(X_test)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f"\n--- {label} | Test ---")
    print(f"Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")
    print(f"AUC (OvR): {test_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report:\n", classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# 1. Original
evaluate_gb_model(X_imputed, y_encoded, "Original (No Resampling)")

# 2. SMOTE
smote = SMOTE(random_state=seed)
X_smote, y_smote = smote.fit_resample(X_imputed, y_encoded)
evaluate_gb_model(X_smote, y_smote, "SMOTE Resampling")

# 3. RUS
rus = RandomUnderSampler(random_state=seed)
X_rus, y_rus = rus.fit_resample(X_imputed, y_encoded)
evaluate_gb_model(X_rus, y_rus, "Random UnderSampling (RUS)")


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set random seed
seed = 42
random.seed(seed)
np.random.seed(seed)

# Load dataset
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Function to train & evaluate MLP model with AUC
def evaluate_mlp_model(X_data, y_data, label):
    # Split data into train (60%), val (20%), test (20%)
    X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.4, random_state=seed, stratify=y_data)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

    # Initialize and train MLP classifier
    mlp_classifier = MLPClassifier(
        hidden_layer_sizes=(150, 100, 50),
        max_iter=300,
        activation='tanh',
        solver='adam',
        alpha=0.0001,
        learning_rate='constant',
        random_state=seed
    )
    mlp_classifier.fit(X_train, y_train)

    # Validation set evaluation
    y_val_pred = mlp_classifier.predict(X_val)
    y_val_proba = mlp_classifier.predict_proba(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')

    print(f"\n--- {label} | Validation ---")
    print(f"Accuracy: {accuracy_score(y_val, y_val_pred) * 100:.2f}%")
    print(f"AUC (OvR): {val_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
    print("Classification Report:\n", classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

    # Test set evaluation
    y_test_pred = mlp_classifier.predict(X_test)
    y_test_proba = mlp_classifier.predict_proba(X_test)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f"\n--- {label} | Test ---")
    print(f"Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")
    print(f"AUC (OvR): {test_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report:\n", classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# 1. Original data
evaluate_mlp_model(X_imputed, y_encoded, "Original (No Resampling)")

# 2. SMOTE resampling
smote = SMOTE(random_state=seed)
X_smote, y_smote = smote.fit_resample(X_imputed, y_encoded)
evaluate_mlp_model(X_smote, y_smote, "SMOTE Resampling")

# 3. RUS resampling
rus = RandomUnderSampler(random_state=seed)
X_rus, y_rus = rus.fit_resample(X_imputed, y_encoded)
evaluate_mlp_model(X_rus, y_rus, "Random UnderSampling (RUS)")
